# Shipout Analysis
The goal of this project is to reduce the number of shipouts. Each shipout carries a real cost: the price of a shipping label plus the labor hours spent to pack the order. By establishing a baseline level of shipouts we can identify which strategies effectively reduce the quantity of shipouts

In [29]:
import pandas as pd
from datetime import datetime, timedelta

# Data Source
RICS Inventory Detail and Stock Status reports are used as the data source for this project. Both reports were restricted to only the footwear class. Although there are shipouts for other product categories, the strategies for reducing the number of shipouts will center around footwear distribution. Inventory detail report was run from 1/1/2026-6/8/2026. The stock status report was run to evaluate inventory levels on 6/9/2026.

In [30]:
inventory_detail = pd.read_csv("InventoryDetail.csv", dtype={"Group": str})
inventory_detail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 449266 entries, 0 to 449265
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Organization     449266 non-null  object 
 1   Group            0 non-null       object 
 2   Sku              449266 non-null  object 
 3   SkuDescription   449266 non-null  object 
 4   SkuColor         449062 non-null  object 
 5   SkuSupplierCode  449266 non-null  object 
 6   SkuClass         449266 non-null  object 
 7   InventoryStore   449266 non-null  int64  
 8   InventoryDate    449266 non-null  object 
 9   GridColumn       449266 non-null  float64
 10  GridRow          448965 non-null  object 
 11  InventoryType    449266 non-null  object 
 12  Qty              449266 non-null  int64  
 13  LineItemCost     449266 non-null  float64
 14  Comment          218947 non-null  object 
dtypes: float64(2), int64(2), object(11)
memory usage: 51.4+ MB


In [31]:
def get_start_date(date_option: str) -> datetime:
    today = datetime.today()

    if date_option == "month_to_date":
        return datetime(today.year, today.month, 1)

    if date_option == "year_to_date":
        return datetime(today.year, 1, 1)

    if date_option == "quarter_to_date":
        quarter_start_month = ((today.month - 1) // 3) * 3 + 1
        return datetime(today.year, quarter_start_month, 1)

    if date_option == "last_30_days":
        return today - timedelta(days=30)

    if date_option == "last_90_days":
        return today - timedelta(days=90)

    raise ValueError(f"Unknown date option: {date_option}")

In [36]:
def filter_transactions(start_date: str, df: pd.DataFrame) -> pd.DataFrame:
    # start_date has format MM/DD/YYYY
    start_dt = pd.to_datetime(start_date)

    retail_stores = [
        1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
        30, 31, 32, 33
    ]

    df = df.copy()
    df["InventoryDate"] = pd.to_datetime(df["InventoryDate"], format="%m/%d/%Y")

    first_txn = (
        df[df["InventoryStore"].isin(retail_stores)]
        .groupby("InventoryStore")["InventoryDate"]
        .min()
    )

    stores_to_include = first_txn[first_txn <= start_dt].index

    return df[df["InventoryStore"].isin(stores_to_include)]

In [37]:
def compute_shipouts_by_store(df: pd.DataFrame) -> pd.DataFrame:
    transfers = df[df["InventoryType"]=="Transfer In"]
    shipouts = transfers[transfers["Comment"].str.startswith("TO# SHP", na=False)]
    shipouts_by_store = (
        shipouts
            .groupby("InventoryStore")
            .agg(
                shipout_units_requested=("Qty", "sum")
            )
    )

    sales = df[df["InventoryType"]=="Sale"]
    sales_by_store = (
        sales
        .groupby("InventoryStore")
        .agg(total_units_sold=("Qty", "sum"))
    )
    sales_by_store["total_units_sold"] = -1*sales_by_store["total_units_sold"]
    
    combined = pd.merge(sales_by_store, shipouts_by_store, left_index=True, right_index=True)
    combined["shipout_percent_of_sales"] = combined["shipout_units_requested"] / combined["total_units_sold"]    
    return combined

In [38]:
user_option = "year_to_date"
start_date = get_start_date(user_option)
print(start_date)
df_ytd = filter_transactions(start_date, inventory_detail)
shipouts_by_store_ytd = compute_shipouts_by_store(df_ytd)
print(shipouts_by_store_ytd)

2026-01-01 00:00:00
                total_units_sold  shipout_units_requested  \
InventoryStore                                              
1                           7790                     1166   
2                           8374                     1156   
3                           7242                     1380   
4                           4834                      908   
5                           5595                      800   
6                           4088                      779   
7                           5420                     1221   
10                          2835                      430   
11                          2924                      784   
12                          2469                      566   
13                          2572                      654   
14                          3895                      791   
15                           383                       85   
16                          2268                      402   
17  

In [8]:
stock_status = pd.read_csv("stock_status.csv")
on_hand_by_store = (
    stock_status
    .groupby("StoreCode")
    .agg(total_on_hand=("OnHand", "sum"))
)

In [9]:
combined = pd.merge(combined, on_hand_by_store, left_index=True, right_index=True)
combined

NameError: name 'combined' is not defined

In [ ]:
established_combined = combined[~combined.index.isin([31, 32, 9])].copy()
established_combined

In [ ]:
established_combined["turnover"] = established_combined["total_sales"] / established_combined["total_on_hand"]
established_combined

In [ ]:
established_combined["transfer_score"] = established_combined["turnover"]/established_combined["shipout_percent_of_sales"]
established_combined.sort_values(by="transfer_score", ascending=False)
# want a metric to reflect stores that request a lot of shipouts despite having high inventory levels
# unlike sales, would expect shipout requests to be lower for higher inventory
# inventory / number of shipouts

In [ ]:
established_combined["shipout_count"].sum() / established_combined["total_sales"].sum(), established_combined["shipout_count"].sum() / established_combined["total_on_hand"].sum()

In [ ]:
established_combined.to_csv("shipout_analysis.csv", index=True)